# BiLSTM trên dataset v2, đầu vào rút gọn (không field Bro)

Kiến trúc và siêu tham số giữ nguyên bản 06dd. **Chỉ đổi đầu vào (STEP 23, phương án B):** bỏ các field chỉ có ở Bro (`service`, `conn_state`, 7 numeric, `has_missed_bytes`) và `principal_class`; còn `app_family`, `proto`, `resp_port_class`, `image_dir_class`, `delta_t_log1p`. Mục đích: cùng một tập field tính được trên dataset ngoài OpTC (skrghosh, ...).

Dataset: bản v2 đã upload (`v2_network_dl_kaggle_upload`), không build lại shards; split cấp cửa sổ phân tầng theo scenario. Một lần chạy (seed 42). Model selection và threshold chỉ dùng VAL; TEST chỉ chạy một lần sau khi checkpoint và threshold đã chốt.

Lưu ý khi đọc kết quả: các field còn lại chính là các field cho luật `powershell.exe` F1 0.994 ở bản v2 đầy đủ, nên F1 trên OpTC không chứng minh mô hình học thêm điều gì.

In [ ]:
import os, sys, json, shutil
from pathlib import Path
import torch
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## Cấu hình
Tên run cố định: `bilstm_v2_subsetB`.

In [ ]:
RUN_NAME = 'bilstm_v2_subsetB'
print('RUN_NAME =', RUN_NAME)


## Resolve dataset
Notebook xử lý cả trường hợp nested folders được mount trực tiếp hoặc Kaggle CLI đã zip top-level directories.


In [ ]:
from __future__ import annotations

import shutil
from pathlib import Path


def unwrap_same_name_folder(
    root: Path,
    name: str,
) -> Path:
    path = root / name

    if not path.exists():
        raise FileNotFoundError(
            f"Missing folder: {path}"
        )

    while (path / name).is_dir():
        path = path / name

    return path


def resolve_kaggle_dataset_root(
    *,
    input_root: str | Path = "/kaggle/input",
    working_root: str | Path = "/kaggle/working/optc_network_dl",
) -> Path:
    input_root = Path(input_root)

    manifests = list(
        input_root.rglob(
            "dataset_manifest.json"
        )
    )

    if not manifests:
        raise FileNotFoundError(
            "Cannot find dataset_manifest.json under /kaggle/input"
        )

    if len(manifests) > 1:
        raise RuntimeError(
            "Nhieu dataset_manifest.json duoc tim thay (vd. dataset LOSO co 3 holdout). "
            "Chi gan MOT dataset vao notebook:\n"
            + "\n".join(str(m) for m in manifests)
        )

    source_root = None

    for manifest in manifests:
        candidate = manifest.parent

        if (
            candidate
            / "build_config.json"
        ).exists():
            source_root = candidate
            break

    if source_root is None:
        raise RuntimeError(
            "Found dataset_manifest.json but not build_config.json"
        )

    working_root = Path(working_root)

    if working_root.exists():
        shutil.rmtree(working_root)

    working_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    for filename in (
        "dataset_manifest.json",
        "build_config.json",
        "README_DATASET.md",
    ):
        src = source_root / filename

        if src.exists():
            (
                working_root / filename
            ).symlink_to(src)

    for folder in (
        "shards",
        "split",
        "preprocessing",
        "integrity",
    ):
        real_folder = unwrap_same_name_folder(
            source_root,
            folder,
        )

        (
            working_root / folder
        ).symlink_to(
            real_folder,
            target_is_directory=True,
        )

    required = [
        "dataset_manifest.json",
        "build_config.json",
        "split/network_sample_manifest.csv",
        "preprocessing/network_category_maps.json",
        "preprocessing/network_numeric_stats.json",
        "preprocessing/network_preprocess_spec.json",
        "preprocessing/network_preprocess_fingerprint.json",
        "shards/train",
        "shards/val",
        "shards/test",
    ]

    for relative in required:
        path = working_root / relative

        print(
            f"{'PASS' if path.exists() else 'FAIL':4s}",
            relative,
        )

        if not path.exists():
            raise FileNotFoundError(path)

    return working_root


DATASET_ROOT = resolve_kaggle_dataset_root()
print('\nDATASET_ROOT =', DATASET_ROOT)
manifest = json.loads((DATASET_ROOT/'dataset_manifest.json').read_text())
print(json.dumps(manifest['split_stats'], indent=2))


## Chọn field đầu vào (phương án B)
Ghi `preprocessing/feature_subset.json` và sửa spec trong bản làm việc. Không đụng dataset gốc.

In [ ]:
import shutil
KEEP_CAT = ['app_family', 'proto', 'resp_port_class', 'image_dir_class']   # phương án B (STEP 23): bỏ service, conn_state, principal_class, 7 numeric Bro, has_missed_bytes
pre = DATASET_ROOT / 'preprocessing'
if pre.is_symlink():  # trên Kaggle thư mục này là symlink tới /kaggle/input (chỉ đọc) nên chép ra bản ghi được
    real = pre.resolve(); pre.unlink(); shutil.copytree(real, pre)
spec_path = pre / 'network_preprocess_spec.json'
spec = json.loads(spec_path.read_text(encoding='utf-8'))
layout = json.loads((pre / 'preprocess_spec.json').read_text(encoding='utf-8'))['model_input_layout']
all_cat = list(layout['x_cat_columns'])  # thứ tự cột x_cat trong shards (chạy lại cell này vẫn an toàn)
assert all(f in all_cat for f in KEEP_CAT), KEEP_CAT
spec['active_categorical_features'] = list(KEEP_CAT)
spec['active_numerical_features'] = []
spec['feature_subset_note'] = 'STEP 23 phuong an B: chi giu ' + ', '.join(KEEP_CAT) + ' + delta_t_log1p'
spec_path.write_text(json.dumps(spec, indent=2, ensure_ascii=False), encoding='utf-8')
(pre / 'feature_subset.json').write_text(json.dumps({
    'x_cat_columns_all': all_cat, 'keep_categorical': list(KEEP_CAT), 'drop_numeric': True, 'zero_binary': True,
}, indent=2), encoding='utf-8')
print('categorical giu:', KEEP_CAT)
print('numeric: bo het (0 cot); has_missed_bytes: hang 0; delta_t_log1p: giu')


## Write versioned training modules into `/kaggle/working`


In [ ]:
WORK_CODE = Path('/kaggle/working/step05b_code')
WORK_CODE.mkdir(parents=True, exist_ok=True)
(WORK_CODE/'05b_train_bilstm.py').write_text('from __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport math\nimport random\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom sklearn.metrics import (\n    average_precision_score,\n    confusion_matrix,\n    f1_score,\n    precision_recall_curve,\n    precision_score,\n    recall_score,\n    roc_auc_score,\n)\nfrom torch.nn.utils.rnn import pack_padded_sequence\nfrom torch.utils.data import DataLoader\n\nfrom network_dl_dataset_v1_1 import (\n    PackedNetworkDataset,\n    ShardAwareBatchSampler,\n    collate_sequence_batch,\n)\n\nSCRIPT_VERSION = "step05b-bilstm-baseline-v1.0"\n\n\ndef load_json(path: Path) -> Any:\n    return json.loads(path.read_text(encoding="utf-8"))\n\n\ndef dump_json(path: Path, obj: Any) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")\n\n\ndef seed_everything(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef categorical_cardinalities(dataset_root: Path) -> tuple[list[str], list[int]]:\n    spec = load_json(dataset_root / "preprocessing" / "network_preprocess_spec.json")\n    maps = load_json(dataset_root / "preprocessing" / "network_category_maps.json")\n    fields = list(spec["active_categorical_features"])\n    cards = []\n    for field in fields:\n        mapping = maps["fields"][field]["mapping"]\n        cards.append(max(int(v) for v in mapping.values()) + 1)\n    return fields, cards\n\n\ndef numeric_fields(dataset_root: Path) -> list[str]:\n    spec = load_json(dataset_root / "preprocessing" / "network_preprocess_spec.json")\n    return list(spec["active_numerical_features"])\n\n\ndef embedding_dim(cardinality: int) -> int:\n    return int(min(16, max(4, round(math.sqrt(cardinality) * 2))))\n\n\nclass BiLSTMBaseline(nn.Module):\n    def __init__(\n        self,\n        *,\n        categorical_cardinalities: list[int],\n        numerical_dim: int,\n        binary_dim: int = 1,\n        event_dim: int = 64,\n        hidden_dim: int = 64,\n        lstm_layers: int = 1,\n        dropout: float = 0.2,\n    ):\n        super().__init__()\n        self.embeddings = nn.ModuleList([\n            nn.Embedding(cardinality, embedding_dim(cardinality))\n            for cardinality in categorical_cardinalities\n        ])\n        cat_dim = sum(emb.embedding_dim for emb in self.embeddings)\n        raw_event_dim = cat_dim + numerical_dim + numerical_dim + binary_dim + 1\n\n        self.event_encoder = nn.Sequential(\n            nn.Linear(raw_event_dim, event_dim),\n            nn.LayerNorm(event_dim),\n            nn.GELU(),\n            nn.Dropout(dropout),\n        )\n        self.lstm = nn.LSTM(\n            input_size=event_dim,\n            hidden_size=hidden_dim,\n            num_layers=lstm_layers,\n            batch_first=True,\n            bidirectional=True,\n            dropout=dropout if lstm_layers > 1 else 0.0,\n        )\n        self.classifier = nn.Sequential(\n            nn.LayerNorm(hidden_dim * 2),\n            nn.Linear(hidden_dim * 2, hidden_dim),\n            nn.GELU(),\n            nn.Dropout(dropout),\n            nn.Linear(hidden_dim, 1),\n        )\n\n    def forward(\n        self,\n        *,\n        x_cat: torch.Tensor,\n        x_num: torch.Tensor,\n        x_num_missing: torch.Tensor,\n        x_bin: torch.Tensor,\n        delta_t_log1p: torch.Tensor,\n        lengths: torch.Tensor,\n    ) -> torch.Tensor:\n        cat_parts = [embedding(x_cat[:, :, i]) for i, embedding in enumerate(self.embeddings)]\n        event = torch.cat([*cat_parts, x_num, x_num_missing, x_bin, delta_t_log1p], dim=-1)\n        event = self.event_encoder(event)\n        packed = pack_padded_sequence(event, lengths.cpu(), batch_first=True, enforce_sorted=False)\n        _, (h_n, _) = self.lstm(packed)\n        pooled = torch.cat([h_n[-2], h_n[-1]], dim=-1)\n        return self.classifier(pooled).squeeze(-1)\n\n\ndef make_loader(dataset, *, batch_size: int, shuffle: bool, seed: int, num_workers: int, pin_memory: bool):\n    sampler = ShardAwareBatchSampler(\n        dataset,\n        batch_size,\n        shuffle=shuffle,\n        seed=seed,\n        drop_last=False,\n    )\n    loader = DataLoader(\n        dataset,\n        batch_sampler=sampler,\n        num_workers=num_workers,\n        pin_memory=pin_memory,\n        persistent_workers=num_workers > 0,\n        collate_fn=collate_sequence_batch,\n    )\n    return loader, sampler\n\n\ndef move_batch(batch: dict[str, Any], device: torch.device) -> dict[str, Any]:\n    result = dict(batch)\n    for key in ("x_cat", "x_num", "x_num_missing", "x_bin", "delta_t_log1p", "labels", "lengths"):\n        result[key] = batch[key].to(device, non_blocking=True)\n    return result\n\n\ndef binary_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> dict[str, Any]:\n    y_pred = (y_prob >= threshold).astype(np.uint8)\n    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()\n    result = {\n        "threshold": float(threshold),\n        "precision": float(precision_score(y_true, y_pred, zero_division=0)),\n        "recall": float(recall_score(y_true, y_pred, zero_division=0)),\n        "f1": float(f1_score(y_true, y_pred, zero_division=0)),\n        "fpr": float(fp / (fp + tn) if (fp + tn) else 0.0),\n        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),\n        "pr_auc": float(average_precision_score(y_true, y_prob)),\n        "roc_auc": float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else None,\n    }\n    return result\n\n\ndef best_f1_threshold(y_true: np.ndarray, y_prob: np.ndarray) -> tuple[float, dict[str, Any]]:\n    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)\n    if len(thresholds) == 0:\n        return 0.5, binary_metrics(y_true, y_prob, 0.5)\n    p, r = precision[:-1], recall[:-1]\n    denom = p + r\n    f1 = np.divide(2 * p * r, denom, out=np.zeros_like(denom), where=denom > 0)\n    index = int(np.nanargmax(f1))\n    threshold = float(thresholds[index])\n    return threshold, binary_metrics(y_true, y_prob, threshold)\n\n\n@torch.no_grad()\ndef predict(model, loader, *, device, amp: bool, max_batches: int | None = None):\n    model.eval()\n    probs, labels, sample_ids = [], [], []\n    for step, raw_batch in enumerate(loader):\n        if max_batches is not None and step >= max_batches:\n            break\n        batch = move_batch(raw_batch, device)\n        with torch.autocast(\n            device_type=device.type,\n            dtype=torch.float16,\n            enabled=amp and device.type == "cuda",\n        ):\n            logits = model(\n                x_cat=batch["x_cat"],\n                x_num=batch["x_num"],\n                x_num_missing=batch["x_num_missing"],\n                x_bin=batch["x_bin"],\n                delta_t_log1p=batch["delta_t_log1p"],\n                lengths=batch["lengths"],\n            )\n        probs.append(torch.sigmoid(logits).cpu().numpy())\n        labels.append(batch["labels"].cpu().numpy())\n        sample_ids.extend(raw_batch["sample_ids"])\n    return np.concatenate(labels), np.concatenate(probs), sample_ids\n\n\ndef save_predictions(path: Path, *, sample_ids, y_true, y_prob, threshold: float) -> None:\n    y_pred = (y_prob >= threshold).astype(np.uint8)\n    with path.open("w", newline="", encoding="utf-8") as f:\n        w = csv.writer(f)\n        w.writerow(["sample_id", "label", "probability", "prediction"])\n        for sid, y, p, pred in zip(sample_ids, y_true, y_prob, y_pred):\n            w.writerow([sid, int(y), float(p), int(pred)])\n\n\ndef save_curves(output_dir: Path, *, y_true, y_prob, prefix: str) -> None:\n    import matplotlib.pyplot as plt\n    from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay\n\n    fig, ax = plt.subplots()\n    PrecisionRecallDisplay.from_predictions(y_true, y_prob, ax=ax)\n    ax.set_title(f"{prefix.upper()} Precision-Recall")\n    fig.tight_layout()\n    fig.savefig(output_dir / f"{prefix}_pr_curve.png", dpi=150)\n    plt.close(fig)\n\n    if len(np.unique(y_true)) > 1:\n        fig, ax = plt.subplots()\n        RocCurveDisplay.from_predictions(y_true, y_prob, ax=ax)\n        ax.set_title(f"{prefix.upper()} ROC")\n        fig.tight_layout()\n        fig.savefig(output_dir / f"{prefix}_roc_curve.png", dpi=150)\n        plt.close(fig)\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--dataset-root", required=True)\n    p.add_argument("--output-dir", required=True)\n    p.add_argument("--epochs", type=int, default=8)\n    p.add_argument("--batch-size", type=int, default=64)\n    p.add_argument("--event-dim", type=int, default=64)\n    p.add_argument("--hidden-dim", type=int, default=64)\n    p.add_argument("--lstm-layers", type=int, default=1)\n    p.add_argument("--dropout", type=float, default=0.2)\n    p.add_argument("--lr", type=float, default=1e-3)\n    p.add_argument("--weight-decay", type=float, default=1e-4)\n    p.add_argument("--grad-clip", type=float, default=1.0)\n    p.add_argument("--patience", type=int, default=3)\n    p.add_argument("--num-workers", type=int, default=2)\n    p.add_argument("--cache-shards", type=int, default=1)\n    p.add_argument("--seed", type=int, default=42)\n    p.add_argument("--no-amp", action="store_true")\n    p.add_argument("--max-train-batches", type=int, default=None, help="Debug only")\n    p.add_argument("--max-eval-batches", type=int, default=None, help="Debug only")\n    args = p.parse_args()\n\n    dataset_root = Path(args.dataset_root)\n    output_dir = Path(args.output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    seed_everything(args.seed)\n\n    manifest = load_json(dataset_root / "dataset_manifest.json")\n    train_stats = manifest["split_stats"]["train"]\n    train_positive = int(train_stats["positive_samples"])\n    train_negative = int(train_stats["negative_samples"])\n    if train_positive <= 0:\n        raise RuntimeError("TRAIN has no positive samples")\n    pos_weight_value = train_negative / train_positive\n\n    cat_fields, cat_cards = categorical_cardinalities(dataset_root)\n    num_fields = numeric_fields(dataset_root)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    amp = (not args.no_amp) and device.type == "cuda"\n\n    print("=" * 100)\n    print("STEP 05B — BiLSTM baseline")\n    print("=" * 100)\n    print(f"[DATASET] {dataset_root}")\n    print(f"[TRAIN] negative={train_negative:,} | positive={train_positive:,} | pos_weight={pos_weight_value:.6f}")\n    print(f"[CAT] {list(zip(cat_fields, cat_cards))}")\n    print(f"[NUM] {num_fields}")\n    print(f"[DEVICE] {device} | amp={amp}")\n\n    train_ds = PackedNetworkDataset(dataset_root, "train", cache_shards=args.cache_shards)\n    val_ds = PackedNetworkDataset(dataset_root, "val", cache_shards=args.cache_shards)\n    test_ds = PackedNetworkDataset(dataset_root, "test", cache_shards=args.cache_shards)\n    pin = device.type == "cuda"\n\n    train_loader, train_sampler = make_loader(train_ds, batch_size=args.batch_size, shuffle=True, seed=args.seed, num_workers=args.num_workers, pin_memory=pin)\n    val_loader, _ = make_loader(val_ds, batch_size=args.batch_size, shuffle=False, seed=args.seed, num_workers=args.num_workers, pin_memory=pin)\n    test_loader, _ = make_loader(test_ds, batch_size=args.batch_size, shuffle=False, seed=args.seed, num_workers=args.num_workers, pin_memory=pin)\n\n    model = BiLSTMBaseline(\n        categorical_cardinalities=cat_cards,\n        numerical_dim=len(num_fields),\n        event_dim=args.event_dim,\n        hidden_dim=args.hidden_dim,\n        lstm_layers=args.lstm_layers,\n        dropout=args.dropout,\n    ).to(device)\n    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    print(f"[PARAMS] {trainable:,}")\n\n    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value, dtype=torch.float32, device=device))\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)\n    scaler = torch.amp.GradScaler("cuda", enabled=amp)\n\n    config = {\n        "script_version": SCRIPT_VERSION,\n        "dataset_root": str(dataset_root),\n        "dataset_preprocess_fingerprint": (manifest["source"].get("step04c_preprocess_fingerprint") or manifest["source"]["step05f_c_preprocess_fingerprint"]),\n        "seed": args.seed,\n        "epochs": args.epochs,\n        "batch_size": args.batch_size,\n        "event_dim": args.event_dim,\n        "hidden_dim": args.hidden_dim,\n        "lstm_layers": args.lstm_layers,\n        "dropout": args.dropout,\n        "lr": args.lr,\n        "weight_decay": args.weight_decay,\n        "grad_clip": args.grad_clip,\n        "patience": args.patience,\n        "num_workers": args.num_workers,\n        "amp": amp,\n        "loss": "BCEWithLogitsLoss",\n        "train_pos_weight": pos_weight_value,\n        "threshold_policy": "maximize F1 on VAL only",\n        "early_stopping_metric": "VAL PR-AUC",\n        "test_policy": "evaluate once after best epoch + VAL threshold are frozen",\n        "categorical_fields": cat_fields,\n        "categorical_cardinalities": cat_cards,\n        "numerical_fields": num_fields,\n        "trainable_parameters": trainable,\n    }\n    dump_json(output_dir / "config.json", config)\n\n    best_pr_auc = -1.0\n    best_epoch = -1\n    stale_epochs = 0\n    history = []\n    best_path = output_dir / "best_model.pt"\n\n    for epoch in range(1, args.epochs + 1):\n        train_sampler.set_epoch(epoch)\n        model.train()\n        running_loss = 0.0\n        seen_batches = 0\n        started = time.time()\n\n        for step, raw_batch in enumerate(train_loader, start=1):\n            if args.max_train_batches is not None and step > args.max_train_batches:\n                break\n            batch = move_batch(raw_batch, device)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):\n                logits = model(\n                    x_cat=batch["x_cat"],\n                    x_num=batch["x_num"],\n                    x_num_missing=batch["x_num_missing"],\n                    x_bin=batch["x_bin"],\n                    delta_t_log1p=batch["delta_t_log1p"],\n                    lengths=batch["lengths"],\n                )\n                loss = criterion(logits, batch["labels"])\n\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)\n            scaler.step(optimizer)\n            scaler.update()\n\n            running_loss += float(loss.item())\n            seen_batches += 1\n            if step % 200 == 0:\n                print(f"[EPOCH {epoch}] batch={step}/{len(train_loader)} loss={running_loss/seen_batches:.6f}")\n\n        train_loss = running_loss / max(1, seen_batches)\n        val_true, val_prob, _ = predict(model, val_loader, device=device, amp=amp, max_batches=args.max_eval_batches)\n        val_pr_auc = float(average_precision_score(val_true, val_prob))\n        val_threshold, val_metrics = best_f1_threshold(val_true, val_prob)\n        elapsed = time.time() - started\n\n        row = {\n            "epoch": epoch,\n            "train_loss": train_loss,\n            "val_pr_auc": val_pr_auc,\n            "val_best_f1_threshold": val_threshold,\n            "val_precision": val_metrics["precision"],\n            "val_recall": val_metrics["recall"],\n            "val_f1": val_metrics["f1"],\n            "val_fpr": val_metrics["fpr"],\n            "elapsed_seconds": elapsed,\n        }\n        history.append(row)\n        print(f"[EPOCH {epoch}] loss={train_loss:.6f} | val_PR_AUC={val_pr_auc:.6f} | val_F1={val_metrics[\'f1\']:.6f} | thr={val_threshold:.6f} | {elapsed/60:.1f} min")\n\n        if val_pr_auc > best_pr_auc + 1e-8:\n            best_pr_auc = val_pr_auc\n            best_epoch = epoch\n            stale_epochs = 0\n            torch.save({\n                "model_state_dict": model.state_dict(),\n                "config": config,\n                "best_epoch": best_epoch,\n                "best_val_pr_auc": best_pr_auc,\n            }, best_path)\n            print(f"[CHECKPOINT] epoch={epoch}")\n        else:\n            stale_epochs += 1\n\n        if stale_epochs >= args.patience:\n            print(f"[EARLY STOP] no VAL PR-AUC improvement for {stale_epochs} epochs")\n            break\n\n    if not history or not best_path.exists():\n        raise RuntimeError("Training did not produce a best checkpoint")\n\n    with (output_dir / "history.csv").open("w", newline="", encoding="utf-8") as f:\n        w = csv.DictWriter(f, fieldnames=list(history[0].keys()))\n        w.writeheader()\n        w.writerows(history)\n\n    checkpoint = torch.load(best_path, map_location=device, weights_only=False)\n    model.load_state_dict(checkpoint["model_state_dict"])\n\n    val_true, val_prob, val_ids = predict(model, val_loader, device=device, amp=amp, max_batches=args.max_eval_batches)\n    threshold, val_metrics = best_f1_threshold(val_true, val_prob)\n    threshold_payload = {\n        "selected_on": "val",\n        "policy": "maximize F1",\n        "threshold": threshold,\n        "best_epoch": best_epoch,\n        "best_val_pr_auc": best_pr_auc,\n        "val_metrics_at_threshold": val_metrics,\n    }\n    dump_json(output_dir / "threshold.json", threshold_payload)\n    save_predictions(output_dir / "val_predictions.csv", sample_ids=val_ids, y_true=val_true, y_prob=val_prob, threshold=threshold)\n    save_curves(output_dir, y_true=val_true, y_prob=val_prob, prefix="val")\n\n    # TEST is touched only after checkpoint and threshold are frozen on VAL.\n    test_true, test_prob, test_ids = predict(model, test_loader, device=device, amp=amp, max_batches=args.max_eval_batches)\n    test_metrics = binary_metrics(test_true, test_prob, threshold)\n    save_predictions(output_dir / "test_predictions.csv", sample_ids=test_ids, y_true=test_true, y_prob=test_prob, threshold=threshold)\n    save_curves(output_dir, y_true=test_true, y_prob=test_prob, prefix="test")\n\n    final_metrics = {\n        "script_version": SCRIPT_VERSION,\n        "best_epoch": best_epoch,\n        "best_val_pr_auc": best_pr_auc,\n        "threshold_selected_on_val": threshold,\n        "val": val_metrics,\n        "test": test_metrics,\n    }\n    dump_json(output_dir / "metrics.json", final_metrics)\n\n    print("\\n" + "=" * 100)\n    print("[SUCCESS] BiLSTM baseline complete")\n    print(f"[BEST EPOCH] {best_epoch}")\n    print(f"[VAL]  PR-AUC={val_metrics[\'pr_auc\']:.6f} | F1={val_metrics[\'f1\']:.6f} | Recall={val_metrics[\'recall\']:.6f} | FPR={val_metrics[\'fpr\']:.6f}")\n    print(f"[TEST] PR-AUC={test_metrics[\'pr_auc\']:.6f} | F1={test_metrics[\'f1\']:.6f} | Recall={test_metrics[\'recall\']:.6f} | FPR={test_metrics[\'fpr\']:.6f}")\n    print(f"[OUTPUT] {output_dir}")\n    print("=" * 100)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
(WORK_CODE/'network_dl_dataset_v1_1.py').write_text('from __future__ import annotations\n\nimport csv\nimport json\nimport random\nfrom collections import OrderedDict, defaultdict\nfrom pathlib import Path\nfrom typing import Any, Iterator\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import Dataset, Sampler\n\n\nclass PackedNetworkDataset(Dataset):\n    def __init__(self, dataset_root: str | Path, split: str, *, cache_shards: int = 2):\n        self.dataset_root = Path(dataset_root)\n        self.split = split.lower().strip()\n        self.cache_shards = max(1, int(cache_shards))\n        self.app_family_col = None\n        self.cat_idx = None\n        self.drop_numeric = False\n        self.zero_binary = False\n        subset_file = self.dataset_root / "preprocessing" / "feature_subset.json"\n        if subset_file.exists():\n            cfg = json.loads(subset_file.read_text(encoding="utf-8"))\n            order = cfg["x_cat_columns_all"]\n            self.cat_idx = [order.index(f) for f in cfg["keep_categorical"]]\n            self.drop_numeric = bool(cfg["drop_numeric"])\n            self.zero_binary = bool(cfg["zero_binary"])\n\n        if self.split not in {"train", "val", "test"}:\n            raise ValueError("split must be train/val/test")\n\n        manifest = self.dataset_root / "split" / "network_sample_manifest.csv"\n        if not manifest.exists():\n            raise FileNotFoundError(manifest)\n\n        self.rows: list[dict[str, Any]] = []\n        with manifest.open("r", newline="", encoding="utf-8") as f:\n            for row in csv.DictReader(f):\n                if row["split"] != self.split:\n                    continue\n                self.rows.append({\n                    "sample_id": row["sample_id"],\n                    "label": int(row["label"]),\n                    "event_count": int(row["event_count"]),\n                    "shard_file": row["shard_file"],\n                    "shard_sample_index": int(row["shard_sample_index"]),\n                })\n\n        if not self.rows:\n            raise RuntimeError(f"No samples found for split={self.split}")\n\n        self._cache: OrderedDict[str, dict[str, np.ndarray]] = OrderedDict()\n\n    def __len__(self) -> int:\n        return len(self.rows)\n\n    def _load_shard(self, relative_path: str) -> dict[str, np.ndarray]:\n        cached = self._cache.get(relative_path)\n        if cached is not None:\n            self._cache.move_to_end(relative_path)\n            return cached\n\n        path = self.dataset_root / relative_path\n        if not path.exists():\n            raise FileNotFoundError(path)\n\n        with np.load(path, allow_pickle=False) as z:\n            shard = {key: z[key] for key in z.files}\n\n        self._cache[relative_path] = shard\n        self._cache.move_to_end(relative_path)\n        while len(self._cache) > self.cache_shards:\n            self._cache.popitem(last=False)\n        return shard\n\n    def _x_cat(self, x):\n        if self.cat_idx is None:\n            return x\n        return x[:, self.cat_idx]  # fancy indexing tao ban sao, shard cache giu nguyen\n\n    def _x_num(self, x):\n        return x[:, :0] if self.drop_numeric else x\n\n    def _x_bin(self, x):\n        return np.zeros_like(x) if self.zero_binary else x\n\n    def __getitem__(self, index: int) -> dict[str, Any]:\n        row = self.rows[index]\n        shard = self._load_shard(row["shard_file"])\n        i = row["shard_sample_index"]\n        offsets = shard["sample_offsets"]\n        start = int(offsets[i])\n        end = int(offsets[i + 1])\n\n        if end - start != row["event_count"]:\n            raise RuntimeError(f"Manifest/shard event_count mismatch: {row[\'sample_id\']}")\n\n        return {\n            "sample_id": row["sample_id"],\n            "label": int(shard["sample_labels"][i]),\n            "event_count": end - start,\n            "x_cat": self._x_cat(shard["x_cat"][start:end]),\n            "x_num": self._x_num(shard["x_num"][start:end]),\n            "x_num_missing": self._x_num(shard["x_num_missing"][start:end]),\n            "x_bin": self._x_bin(shard["x_bin"][start:end]),\n            "delta_t_log1p": shard["delta_t_log1p"][start:end],\n        }\n\n\nclass ShardAwareBatchSampler(Sampler[list[int]]):\n    def __init__(self, dataset: PackedNetworkDataset, batch_size: int, *, shuffle: bool, seed: int = 42, drop_last: bool = False):\n        self.dataset = dataset\n        self.batch_size = int(batch_size)\n        self.shuffle = bool(shuffle)\n        self.seed = int(seed)\n        self.drop_last = bool(drop_last)\n        self.epoch = 0\n        if self.batch_size <= 0:\n            raise ValueError("batch_size must be > 0")\n\n        grouped: dict[str, list[int]] = defaultdict(list)\n        for idx, row in enumerate(dataset.rows):\n            grouped[row["shard_file"]].append(idx)\n        self.groups = list(grouped.items())\n\n        total = 0\n        for _, indices in self.groups:\n            total += len(indices) // self.batch_size if self.drop_last else (len(indices) + self.batch_size - 1) // self.batch_size\n        self._len = total\n\n    def set_epoch(self, epoch: int) -> None:\n        self.epoch = int(epoch)\n\n    def __len__(self) -> int:\n        return self._len\n\n    def __iter__(self) -> Iterator[list[int]]:\n        rng = random.Random(self.seed + self.epoch)\n        groups = [(shard, list(indices)) for shard, indices in self.groups]\n        if self.shuffle:\n            rng.shuffle(groups)\n\n        for _, indices in groups:\n            if self.shuffle:\n                rng.shuffle(indices)\n            for start in range(0, len(indices), self.batch_size):\n                batch = indices[start:start + self.batch_size]\n                if self.drop_last and len(batch) < self.batch_size:\n                    continue\n                yield batch\n\n\ndef collate_sequence_batch(batch: list[dict[str, Any]]) -> dict[str, Any]:\n    if not batch:\n        raise ValueError("Empty batch")\n\n    lengths = torch.tensor([x["event_count"] for x in batch], dtype=torch.long)\n    max_len = int(lengths.max().item())\n    bsz = len(batch)\n    c = batch[0]["x_cat"].shape[1]\n    n = batch[0]["x_num"].shape[1]\n    binary_dim = batch[0]["x_bin"].shape[1]\n\n    x_cat = torch.zeros(bsz, max_len, c, dtype=torch.long)\n    x_num = torch.zeros(bsz, max_len, n, dtype=torch.float32)\n    x_missing = torch.zeros(bsz, max_len, n, dtype=torch.float32)\n    x_bin = torch.zeros(bsz, max_len, binary_dim, dtype=torch.float32)\n    delta = torch.zeros(bsz, max_len, 1, dtype=torch.float32)\n    mask = torch.zeros(bsz, max_len, dtype=torch.bool)\n    labels = torch.tensor([x["label"] for x in batch], dtype=torch.float32)\n\n    for i, item in enumerate(batch):\n        t = item["event_count"]\n        x_cat[i, :t] = torch.from_numpy(np.asarray(item["x_cat"], dtype=np.int64))\n        x_num[i, :t] = torch.from_numpy(np.asarray(item["x_num"], dtype=np.float32))\n        x_missing[i, :t] = torch.from_numpy(np.asarray(item["x_num_missing"], dtype=np.float32))\n        x_bin[i, :t] = torch.from_numpy(np.asarray(item["x_bin"], dtype=np.float32))\n        delta[i, :t, 0] = torch.from_numpy(np.asarray(item["delta_t_log1p"], dtype=np.float32))\n        mask[i, :t] = True\n\n    return {\n        "x_cat": x_cat,\n        "x_num": x_num,\n        "x_num_missing": x_missing,\n        "x_bin": x_bin,\n        "delta_t_log1p": delta,\n        "attention_mask": mask,\n        "labels": labels,\n        "lengths": lengths,\n        "sample_ids": [x["sample_id"] for x in batch],\n    }\n', encoding='utf-8')
print('Wrote:', sorted(p.name for p in WORK_CODE.iterdir()))
sys.path.insert(0, str(WORK_CODE))

## DataLoader sanity
Một batch trước khi full training.


In [ ]:
from network_dl_dataset_v1_1 import PackedNetworkDataset, ShardAwareBatchSampler, collate_sequence_batch
from torch.utils.data import DataLoader
ds = PackedNetworkDataset(DATASET_ROOT, 'train', cache_shards=1)
sampler = ShardAwareBatchSampler(ds, 8, shuffle=False)
loader = DataLoader(ds, batch_sampler=sampler, num_workers=0, collate_fn=collate_sequence_batch)
b = next(iter(loader))
print('samples:', len(ds))
print('x_cat:', tuple(b['x_cat'].shape))
print('x_num:', tuple(b['x_num'].shape))
print('lengths:', b['lengths'].tolist())
print('app_family ids in batch:', sorted(set(b['x_cat'][:, :, 0].unique().tolist()))[:10])


## Train BiLSTM baseline
`BCEWithLogitsLoss(pos_weight=TRAIN_NEG/TRAIN_POS)`, early stopping theo VAL PR-AUC, threshold maximize F1 trên VAL.


In [ ]:
OUTPUT_DIR = Path(f'/kaggle/working/{RUN_NAME}')
cmd = (
    f"python {WORK_CODE/'05b_train_bilstm.py'} "
    f"--dataset-root '{DATASET_ROOT}' "
    f"--output-dir '{OUTPUT_DIR}' "
    "--epochs 8 --batch-size 64 --event-dim 64 --hidden-dim 64 "
    "--lstm-layers 1 --dropout 0.2 --lr 0.001 --weight-decay 0.0001 "
    "--patience 3 --num-workers 2"
)
print(cmd)
import subprocess
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f'Training failed with exit code {ret}')


## Final metrics


In [ ]:
metrics = json.loads((OUTPUT_DIR/'metrics.json').read_text())
threshold = json.loads((OUTPUT_DIR/'threshold.json').read_text())
print(json.dumps(metrics, indent=2))
print(json.dumps(threshold, indent=2))


## Zip training outputs


In [ ]:
archive = shutil.make_archive(f'/kaggle/working/{RUN_NAME}', 'zip', OUTPUT_DIR)
print('Created:', archive)
